In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"

# Existing 17-feature training data
feature_df = pd.read_csv(
    RESULTS_DIR / "training_features.csv"
)

print("Training features:", feature_df.shape)
print(feature_df.columns.tolist())

Training features: (1259849, 19)
['name_exact', 'name_token_jaccard', 'name_char_similarity', 'name_token_set_ratio', 'name_token_sort_ratio', 'name_shared_tokens', 'name_length_ratio', 'address_exact', 'address_token_jaccard', 'address_char_similarity', 'address_token_set_ratio', 'address_token_sort_ratio', 'address_shared_tokens', 'address_length_ratio', 'country_match', 'name_missing', 'address_missing', 'label', 'source1_entity_id']


In [2]:
TRAIN_DIR = PROJECT_ROOT / "student_resource"/"dataset" / "train"

s1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t",
    dtype=str
)

s2 = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t",
    dtype=str
)

s3 = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t",
    dtype=str
)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)

S1: (2206821, 4)
S2: (5034616, 4)
S3: (5285603, 4)


In [3]:
candidate_source = pd.concat(
    [
        s2[[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]],
        s3[[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]]
    ],
    ignore_index=True
)

candidate_source = candidate_source.set_index("entity_id")

s1_lookup = s1.set_index("entity_id")

print("Candidate source:", candidate_source.shape)

Candidate source: (10320219, 3)


In [5]:
print("Columns:")
print(feature_df.columns.tolist())

print("\nIndex name:")
print(feature_df.index.name)

print("\nFirst rows:")
display(feature_df.head())

Columns:
['name_exact', 'name_token_jaccard', 'name_char_similarity', 'name_token_set_ratio', 'name_token_sort_ratio', 'name_shared_tokens', 'name_length_ratio', 'address_exact', 'address_token_jaccard', 'address_char_similarity', 'address_token_set_ratio', 'address_token_sort_ratio', 'address_shared_tokens', 'address_length_ratio', 'country_match', 'name_missing', 'address_missing', 'label', 'source1_entity_id']

Index name:
None

First rows:


,name_exact,name_token_jaccard,name_char_similarity,name_token_set_ratio,name_token_sort_ratio,name_shared_tokens,name_length_ratio,address_exact,address_token_jaccard,address_char_similarity,address_token_set_ratio,address_token_sort_ratio,address_shared_tokens,address_length_ratio,country_match,name_missing,address_missing,label,source1_entity_id
0,0,0.750000,0.866667,1.000000,0.866667,3,0.764706,0,1.000000,0.880000,1.000000,1.000000,5,1.000000,1,0,0,1,S1-976369483
1,0,0.166667,0.342857,0.457143,0.457143,1,0.590909,0,0.000000,0.380952,0.380952,0.380952,0,0.657895,1,0,0,0,S1-976369483
2,0,0.166667,0.444444,0.400000,0.266667,1,0.406250,0,0.000000,0.360000,0.320000,0.320000,0,1.000000,1,0,0,0,S1-976369483
3,0,0.142857,0.363636,0.400000,0.327273,1,0.309524,0,0.100000,0.360656,0.426230,0.426230,1,0.694444,1,0,0,0,S1-976369483
4,0,0.166667,0.444444,0.400000,0.400000,1,0.406250,0,0.090909,0.375000,0.406250,0.406250,1,0.641026,1,0,0,0,S1-976369483


In [6]:
print("Variables containing 'pair' or 'candidate':")

for name in sorted(globals()):
    if "pair" in name.lower() or "candidate" in name.lower():
        obj = globals()[name]
        try:
            print(
                name,
                type(obj).__name__,
                getattr(obj, "shape", ""),
                getattr(obj, "columns", "").tolist() if hasattr(getattr(obj, "columns", None), "tolist") else ""
            )
        except:
            print(name, type(obj).__name__)

Variables containing 'pair' or 'candidate':
candidate_source DataFrame (10320219, 3) ['business_name', 'business_address', 'country']


In [7]:
print("All DataFrame-like variables:")
for name in sorted(globals()):
    obj = globals()[name]
    if isinstance(obj, pd.DataFrame):
        print(
            f"{name:30s}",
            obj.shape,
            list(obj.columns)
        )

All DataFrame-like variables:
candidate_source               (10320219, 3) ['business_name', 'business_address', 'country']
feature_df                     (1259849, 19) ['name_exact', 'name_token_jaccard', 'name_char_similarity', 'name_token_set_ratio', 'name_token_sort_ratio', 'name_shared_tokens', 'name_length_ratio', 'address_exact', 'address_token_jaccard', 'address_char_similarity', 'address_token_set_ratio', 'address_token_sort_ratio', 'address_shared_tokens', 'address_length_ratio', 'country_match', 'name_missing', 'address_missing', 'label', 'source1_entity_id']
s1                             (2206821, 4) ['entity_id', 'business_name', 'business_address', 'country']
s1_lookup                      (2206821, 3) ['business_name', 'business_address', 'country']
s2                             (5034616, 4) ['entity_id', 'business_name', 'business_address', 'country']
s3                             (5285603, 4) ['entity_id', 'business_name', 'business_address', 'country']


In [9]:
# Recover the EXACT 10,000 S1 records used to create feature_df

sample_s1_ids = feature_df["source1_entity_id"].unique()

print("Unique S1 IDs:", len(sample_s1_ids))

# Preserve the original S1 dataframe ordering
s1_sample = s1[s1["entity_id"].isin(sample_s1_ids)].copy()

print("S1 sample shape:", s1_sample.shape)
print("Unique S1 IDs:", s1_sample["entity_id"].nunique())

Unique S1 IDs: 9972
S1 sample shape: (9972, 4)
Unique S1 IDs: 9972


In [13]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "code" / "business_entity_resolution" / "src"

sys.path.insert(0, str(SRC_DIR))

from business_entity_resolution.preprocessing.normalization import (
    normalize_name,
    normalize_address,
)

In [18]:
from collections import defaultdict
from itertools import combinations

# Make sure all required functions exist
from business_entity_resolution.preprocessing.normalization import (
    normalize_name,
    normalize_address,
)

# Same token rules used in Notebook 06
NAME_STOPWORDS = {
    "private", "limited", "corporation", "company", "incorporated",
    "llc", "llp", "pllc", "corp", "inc", "co", "group",
    "holding", "holdings", "services", "service", "solutions",
    "enterprises", "international", "global", "partners",
    "associates", "consultancy", "consultants", "and", "india"
}

ADDRESS_STOPWORDS = {
    "road", "street", "avenue", "boulevard", "drive", "lane",
    "highway", "parkway", "court", "place", "trail", "circle",
    "suite", "apartment", "floor", "flat", "unit", "block",
    "plot", "sector", "nagar", "colony", "near", "opposite",
    "opp", "main", "city", "county", "north", "south", "east",
    "west", "the", "and", "for", "null"
}

def informative_name_tokens(text):
    if not text:
        return []
    return [
        t for t in normalize_name(text).split()
        if len(t) >= 3 and t not in NAME_STOPWORDS
    ]

def informative_address_tokens(text):
    if pd.isna(text) or not str(text).strip():
        return []
    return [
        t for t in normalize_address(text).split()
        if len(t) >= 3 and t not in ADDRESS_STOPWORDS
    ]


# Build indexes
name_index = defaultdict(list)
address_index = defaultdict(list)
name_token_index = defaultdict(list)
address_pair_index = defaultdict(list)

for row in candidate_source.itertuples():

    entity_id = row.Index
    country = row.country

    # A: exact normalized name
    name = normalize_name(row.business_name)

    if name:
        name_index[(country, name)].append(entity_id)

    # B: exact normalized address
    address = normalize_address(row.business_address)

    if address:
        address_index[(country, address)].append(entity_id)

    # C: informative name token
    for token in informative_name_tokens(row.business_name):
        name_token_index[(country, token)].append(entity_id)

    # D: address token pairs
    tokens = informative_address_tokens(row.business_address)

    if len(tokens) >= 2:
        for t1, t2 in combinations(sorted(tokens), 2):
            address_pair_index[(country, t1, t2)].append(entity_id)


print("name_index:", len(name_index))
print("address_index:", len(address_index))
print("name_token_index:", len(name_token_index))
print("address_pair_index:", len(address_pair_index))

KeyboardInterrupt: 

In [19]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Ready")

Ready


In [21]:
cand = candidate_source.reset_index().rename(
    columns={"entity_id": "candidate_entity_id"}
)

print(cand.columns.tolist())
print(cand.shape)

# S1 sample
s1_exp = s1_sample[
    ["entity_id", "business_name", "business_address", "country"]
].copy()

s1_exp["norm_name"] = s1_exp["business_name"].map(normalize_name)
s1_exp["norm_address"] = s1_exp["business_address"].map(normalize_address)

cand["norm_name"] = cand["business_name"].map(normalize_name)
cand["norm_address"] = cand["business_address"].map(normalize_address)

# Exact-name candidates
name_pairs = s1_exp.merge(
    cand[["candidate_entity_id", "country", "norm_name"]],
    on=["country", "norm_name"],
    how="inner"
)

print("Name pairs:", len(name_pairs))

# Exact-address candidates
address_pairs = s1_exp.merge(
    cand[["candidate_entity_id", "country", "norm_address"]],
    on=["country", "norm_address"],
    how="inner"
)

print("Address pairs:", len(address_pairs))

['candidate_entity_id', 'business_name', 'business_address', 'country']
(10320219, 4)
Name pairs: 109311
Address pairs: 5000


In [27]:
from pathlib import Path
import pandas as pd

TRAIN_DIR = PROJECT_ROOT / "student_resource"/"dataset" / "train"

# Find the ground-truth TSV automatically
gt_files = list(TRAIN_DIR.glob("*.tsv"))

print("TSV files:")
for f in gt_files:
    print(f.name)

TSV files:
train_ground_truth.tsv
train_source1.tsv
train_source2.tsv
train_source3.tsv


In [28]:
# Load training ground truth

gt_path = PROJECT_ROOT / "student_resource"/ "dataset" / "train" / "train_ground_truth.tsv"

ground_truth_df = pd.read_csv(
    gt_path,
    sep="\t",
    dtype=str
)

print("Ground truth shape:", ground_truth_df.shape)
print(ground_truth_df.head())
print(ground_truth_df.columns.tolist())

Ground truth shape: (2206821, 2)
  source1_entity_id                                 matched_entity_ids
0         S1-965667  S2-681193310,S2-743505751,S3-775321672,S3-1129...
1       S1-55344266  S2-249013014,S2-197070651,S3-478195123,S3-3843...
2      S1-343815751             S2-790675320,S2-479876582,S3-878454467
3      S1-656753428              S2-153058913,S2-24659151,S3-679606215
4      S1-102811957  S2-478959098,S2-553508714,S2-625774905,S3-7280...
['source1_entity_id', 'matched_entity_ids']


In [29]:
ground_truth = {}

for row in ground_truth_df.itertuples(index=False):
    if pd.isna(row.matched_entity_ids) or not row.matched_entity_ids:
        ground_truth[row.source1_entity_id] = set()
    else:
        ground_truth[row.source1_entity_id] = set(
            row.matched_entity_ids.split(",")
        )

print("S1 entities in ground truth:", len(ground_truth))

S1 entities in ground truth: 2206821


In [30]:
positive_keys = {
    (source1_id, candidate_id)
    for source1_id, matched_ids in ground_truth.items()
    for candidate_id in matched_ids
}

candidate_pairs_tfidf["label"] = [
    int((s1_id, cand_id) in positive_keys)
    for s1_id, cand_id in zip(
        candidate_pairs_tfidf["source1_entity_id"],
        candidate_pairs_tfidf["candidate_entity_id"]
    )
]

print("Candidate pairs:", len(candidate_pairs_tfidf))
print(candidate_pairs_tfidf["label"].value_counts())

Candidate pairs: 113490
label
0    102032
1     11458
Name: count, dtype: int64


In [31]:
print("Total GT positive pairs:", len(positive_keys))

captured = int(candidate_pairs_tfidf["label"].sum())

print("Captured positive pairs:", captured)
print("Pair recall:", captured / len(positive_keys))

Total GT positive pairs: 7638365
Captured positive pairs: 11458
Pair recall: 0.0015000592404264526


In [32]:
# Correct recall: only GT pairs belonging to our sampled S1 entities

sample_ids = set(s1_sample["entity_id"])

sample_positive_keys = {
    (source1_id, candidate_id)
    for source1_id, matched_ids in ground_truth.items()
    if source1_id in sample_ids
    for candidate_id in matched_ids
}

captured = int(candidate_pairs_tfidf["label"].sum())

print("Sampled S1 entities:", len(sample_ids))
print("GT positive pairs for sampled S1s:", len(sample_positive_keys))
print("Captured positive pairs:", captured)
print("Pair recall:", captured / len(sample_positive_keys))

Sampled S1 entities: 9972
GT positive pairs for sampled S1s: 34427
Captured positive pairs: 11458
Pair recall: 0.332820170215238


In [33]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"

feature_df = pd.read_csv(
    RESULTS_DIR / "training_features.csv"
)

print("Shape:", feature_df.shape)
print(feature_df.columns.tolist())

Shape: (1259849, 19)
['name_exact', 'name_token_jaccard', 'name_char_similarity', 'name_token_set_ratio', 'name_token_sort_ratio', 'name_shared_tokens', 'name_length_ratio', 'address_exact', 'address_token_jaccard', 'address_char_similarity', 'address_token_set_ratio', 'address_token_sort_ratio', 'address_shared_tokens', 'address_length_ratio', 'country_match', 'name_missing', 'address_missing', 'label', 'source1_entity_id']


In [35]:
print("training_pairs" in globals())
print("candidate_rows" in globals())

if "training_pairs" in globals():
    print(training_pairs.shape)
    print(training_pairs.columns.tolist())

if "candidate_rows" in globals():
    print(len(candidate_rows))

False
False


In [34]:
# Make sure entity IDs are strings
feature_df["source1_entity_id"] = feature_df["source1_entity_id"].astype(str)
feature_df["candidate_entity_id"] = feature_df["candidate_entity_id"].astype(str)

# Lookups
s1_lookup = s1.set_index("entity_id")
candidate_lookup = candidate_source.copy()

candidate_lookup.index = candidate_lookup.index.astype(str)

# Map S1 text
feature_df["s1_name"] = (
    feature_df["source1_entity_id"]
    .map(s1_lookup["business_name"])
    .fillna("")
)

feature_df["cand_name"] = (
    feature_df["candidate_entity_id"]
    .map(candidate_lookup["business_name"])
    .fillna("")
)

print(feature_df[[
    "source1_entity_id",
    "candidate_entity_id",
    "s1_name",
    "cand_name",
    "label"
]].head())

KeyError: 'candidate_entity_id'